# Exercise 2 — RSI Mean Reversion

RSI mean reversion bets that an oversold market will bounce back. When RSI falls below 30 (too much selling), go long. When it rises above 70 (too much buying), go flat. Between the thresholds, hold the previous position — avoiding whipsaw from small RSI fluctuations.

In [ ]:
import pandas as pd, math, warnings

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })

def _sma(s, w):  return s.rolling(w).mean()
def _ema(s, w):  return s.ewm(span=w, adjust=False).mean()
def _rsi(s, w):
    d = s.diff()
    g = d.clip(lower=0).rolling(w).mean()
    l = (-d.clip(upper=0)).rolling(w).mean()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        rs = g / l
    return 100 - (100 / (1 + rs))
def sma_crossover(df, fast=20, slow=50):
    close = df["Close"]
    return (_sma(close, fast) > _sma(close, slow)).fillna(False).astype(int)

def rsi_mean_reversion(df, window=14, oversold=30, overbought=70):
    """Long when RSI < oversold; flat when RSI > overbought; hold otherwise.

    Steps:
      1. rsi_s  = _rsi(df["Close"], window)
      2. signal = pd.Series(float("nan"), index=df.index)
      3. signal[rsi_s < oversold]   = 1.0
      4. signal[rsi_s > overbought] = 0.0
      5. return signal.ffill().fillna(0).astype(int)

    The ffill() carries the last explicit 0 or 1 through the neutral zone.
    The final fillna(0) handles the warmup period (before first RSI value).
    """
    # TODO: implement the 5 steps above
    return pd.Series(0, index=df.index)


### Checks

In [ ]:
checks = 0

# 1 — returns same-length Series, no NaN, values in {0,1}
try:
    df  = _synthetic()
    sig = rsi_mean_reversion(df)
    assert isinstance(sig, pd.Series) and len(sig) == len(df)
    assert not sig.isna().any()
    assert set(sig.unique()).issubset({0, 1})
    checks += 1; print("✅ 1 valid Series: same length, no NaN, values in {0,1}")
except Exception as e:
    print("❌ 1:", e)

# 2 — with loose thresholds, produces both 0s and 1s
try:
    sig = rsi_mean_reversion(_synthetic(), window=14, oversold=40, overbought=60)
    assert (sig == 1).any(), "no 1s with oversold=40"
    assert (sig == 0).any(), "no 0s with overbought=60"
    checks += 1; print("✅ 2 produces both 0s and 1s with loose thresholds")
except Exception as e:
    print("❌ 2:", e)

# 3 — when RSI is forced below oversold, signal = 1
try:
    df    = _synthetic()
    close = df["Close"]
    # Compute RSI to find an oversold bar
    d = close.diff()
    g = d.clip(lower=0).rolling(14).mean()
    l = (-d.clip(upper=0)).rolling(14).mean()
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        rs = g / l
    rsi_s = 100 - (100 / (1 + rs))
    oversold_mask = rsi_s < 30
    if oversold_mask.any():
        sig = rsi_mean_reversion(df, window=14, oversold=30, overbought=70)
        assert (sig[oversold_mask] == 1).all(), "oversold bars should give signal=1"
    checks += 1; print("✅ 3 RSI < oversold → signal = 1")
except Exception as e:
    print("❌ 3:", e)

# 4 — when RSI > overbought, signal = 0
try:
    df    = _synthetic()
    close = df["Close"]
    d = close.diff()
    g = d.clip(lower=0).rolling(14).mean()
    l = (-d.clip(upper=0)).rolling(14).mean()
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        rs = g / l
    rsi_s = 100 - (100 / (1 + rs))
    overbought_mask = rsi_s > 70
    if overbought_mask.any():
        sig = rsi_mean_reversion(df, window=14, oversold=30, overbought=70)
        assert (sig[overbought_mask] == 0).all(), "overbought bars should give signal=0"
    checks += 1; print("✅ 4 RSI > overbought → signal = 0")
except Exception as e:
    print("❌ 4:", e)

# 5 — default period: signal is 0 for first window rows
try:
    df  = _synthetic()
    sig = rsi_mean_reversion(df, window=14)
    # Before first RSI value, signal should be 0 (from fillna(0))
    assert sig.iloc[0] == 0, f"expected 0 before first RSI, got {sig.iloc[0]}"
    checks += 1; print("✅ 5 warmup period (before first RSI) gives signal=0")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
